# XLM-RoBERTa-base — Subtask 1 veracity classifier (corrected comparison run)

This notebook is **identical across the four compared models** (MuRIL, IndicBERTv2, mBERT, XLM-R). Only `MODEL_NAME` / `TAG` in the config cell differ.

Fixes compared with the earlier notebooks:
1. `greater_is_better=True` for `macro_f1` (it was `False`, so the *worst* epoch was being selected and early stopping fired too soon).
2. `set_seed(seed)` is called **before** the model is created, so the classifier head initialisation is reproducible.
3. Same early-stopping patience (3) for every model.
4. 3 seeds per model; results are reported as mean ± std.
5. `token_type_ids` are kept for BERT-style models (they were silently dropped before). Set `USE_TOKEN_TYPE_IDS = False` to reproduce the old behaviour.
6. Checkpoints go to local disk, are unique per model/seed, and are deleted after each run (no more shared `checkpoint_muril` folder on Drive).

All four notebooks must read the **same** `train_subtask1.json` / `dev_subtask1.json`.


In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
!pip uninstall -y torchvision
!pip install -q transformers datasets accelerate scikit-learn pandas

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# ---------------- the ONLY two lines that differ between the 4 notebooks ----------------
MODEL_NAME = "xlm-roberta-base"
TAG        = "xlmr"
# -----------------------------------------------------------------------------------------

# Data: must be the same files for all four models
DRIVE_FOLDER = "/content/drive/MyDrive/Anindita_Fire"
TRAIN_FILE   = os.path.join(DRIVE_FOLDER, "train_subtask1.json")
DEV_FILE     = os.path.join(DRIVE_FOLDER, "dev_subtask1.json")

# Results (small JSON) go to Drive; heavy checkpoints stay on local disk and are deleted after each run
RESULTS_DIR = os.path.join(DRIVE_FOLDER, "results_model_comparison")
CKPT_ROOT   = "/content/ckpt"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CKPT_ROOT, exist_ok=True)

# Shared hyperparameters (identical for all models)
SEEDS          = [42, 43, 44]
MAX_LEN        = 256
BATCH_SIZE     = 16
LEARNING_RATE  = 2e-5
WEIGHT_DECAY   = 0.01
NUM_EPOCHS     = 6
WARMUP_RATIO   = 0.1
PATIENCE       = 3
USE_TOKEN_TYPE_IDS = True

In [ ]:
import json
from datasets import Dataset

with open(TRAIN_FILE, encoding="utf-8") as f:
    train_raw = json.load(f)
with open(DEV_FILE, encoding="utf-8") as f:
    dev_raw = json.load(f)

label2id = {"SUPPORTS": 0, "REFUTES": 1}
id2label = {0: "SUPPORTS", 1: "REFUTES"}

def to_hf_format(raw):
    return {
        "claim":    [x["Text"] for x in raw],
        "evidence": [x["Evidence"] for x in raw],
        "label":    [label2id[x["Label"]] for x in raw],
    }

train_ds = Dataset.from_dict(to_hf_format(train_raw))
dev_ds   = Dataset.from_dict(to_hf_format(dev_raw))
print("train:", len(train_ds), "| dev:", len(dev_ds))   # expected: 4536 | 1134

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(batch["claim"], batch["evidence"],
                     truncation=True, max_length=MAX_LEN, padding="max_length")

train_tok = train_ds.map(tokenize_fn, batched=True)
dev_tok   = dev_ds.map(tokenize_fn, batched=True)

cols = ["input_ids", "attention_mask", "label"]
if USE_TOKEN_TYPE_IDS and "token_type_ids" in train_tok.column_names:
    cols.insert(2, "token_type_ids")
print("Model inputs used:", cols)

train_tok.set_format(type="torch", columns=cols)
dev_tok.set_format(type="torch", columns=cols)

In [ ]:
import gc, shutil
import numpy as np
import torch
import torch.nn as nn
from collections import Counter
from sklearn.metrics import f1_score, accuracy_score, precision_recall_fscore_support
from transformers import (AutoModelForSequenceClassification, Trainer, TrainingArguments,
                          EarlyStoppingCallback, set_seed)

# class weights from the training label distribution (same for every model)
counts = Counter(train_ds["label"])
total = sum(counts.values())
class_weights = torch.tensor([total / (2 * counts[0]), total / (2 * counts[1])], dtype=torch.float)
print("Class weights:", class_weights)   # expected: tensor([0.7461, 1.5160])

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss_fct = nn.CrossEntropyLoss(weight=class_weights.to(outputs.logits.device))
        loss = loss_fct(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    _, _, f1_per_class, _ = precision_recall_fscore_support(labels, preds, average=None, zero_division=0)
    return {
        "macro_f1":    f1_score(labels, preds, average="macro"),
        "accuracy":    accuracy_score(labels, preds),
        "f1_supports": f1_per_class[0],
        "f1_refutes":  f1_per_class[1],
    }

In [ ]:
def run_one_seed(seed):
    set_seed(seed)   # BEFORE the model is created -> reproducible classifier-head init

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2, id2label=id2label, label2id=label2id
    )

    steps_per_epoch = len(train_tok) // BATCH_SIZE
    total_steps     = steps_per_epoch * NUM_EPOCHS
    warmup_steps    = int(WARMUP_RATIO * total_steps)

    out_dir = os.path.join(CKPT_ROOT, f"{TAG}_seed{seed}")
    args = TrainingArguments(
        output_dir=out_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,            # <-- FIX: higher macro-F1 is better
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        warmup_steps=warmup_steps,
        seed=seed,
        report_to="none",
    )

    trainer = WeightedTrainer(
        model=model,
        args=args,
        train_dataset=train_tok,
        eval_dataset=dev_tok,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
    )
    trainer.train()

    # per-epoch log
    epoch_log = [
        {"epoch": int(round(h["epoch"])), "eval_loss": h["eval_loss"],
         "macro_f1": h["eval_macro_f1"], "accuracy": h["eval_accuracy"],
         "f1_supports": h["eval_f1_supports"], "f1_refutes": h["eval_f1_refutes"]}
        for h in trainer.state.log_history if "eval_macro_f1" in h
    ]
    best_log = max(epoch_log, key=lambda r: r["macro_f1"])

    # metrics of the model that Trainer actually loaded at the end (should be the best-F1 epoch)
    final = trainer.evaluate()
    if abs(final["eval_macro_f1"] - best_log["macro_f1"]) > 1e-3:
        print("WARNING: loaded checkpoint F1 != best logged F1 -> check checkpoint selection")

    result = {
        "seed": seed,
        "epochs_run": len(epoch_log),
        "best_epoch": best_log["epoch"],
        "macro_f1": float(final["eval_macro_f1"]),
        "accuracy": float(final["eval_accuracy"]),
        "f1_supports": float(final["eval_f1_supports"]),
        "f1_refutes": float(final["eval_f1_refutes"]),
        "epoch_log": epoch_log,
    }

    # free GPU + disk before the next seed
    del trainer, model
    gc.collect(); torch.cuda.empty_cache()
    shutil.rmtree(out_dir, ignore_errors=True)
    return result


all_results = []
for seed in SEEDS:
    print(f"\n{'='*20} {TAG} | seed {seed} {'='*20}")
    all_results.append(run_one_seed(seed))

In [ ]:
import pandas as pd

df = pd.DataFrame([{k: v for k, v in r.items() if k != "epoch_log"} for r in all_results])
print(f"Per-seed results for {MODEL_NAME}")
display(df.round(4))

metrics = ["macro_f1", "accuracy", "f1_supports", "f1_refutes"]
summary = {m: {"mean": float(df[m].mean()), "std": float(df[m].std(ddof=1))} for m in metrics}

print("\nMean ± std over seeds", SEEDS, "(sample std, ddof=1)")
for m in metrics:
    print(f"  {m:12s} {summary[m]['mean']:.3f} ± {summary[m]['std']:.3f}")
print("  best epochs per seed:", df["best_epoch"].tolist(), "| epochs run:", df["epochs_run"].tolist())

out = {
    "model": MODEL_NAME, "tag": TAG, "seeds": SEEDS,
    "config": {"max_len": MAX_LEN, "batch_size": BATCH_SIZE, "lr": LEARNING_RATE,
               "weight_decay": WEIGHT_DECAY, "num_epochs": NUM_EPOCHS,
               "warmup_ratio": WARMUP_RATIO, "patience": PATIENCE,
               "token_type_ids": USE_TOKEN_TYPE_IDS, "greater_is_better": True},
    "summary": summary,
    "runs": all_results,
}
path = os.path.join(RESULTS_DIR, f"results_{TAG}.json")
with open(path, "w", encoding="utf-8") as f:
    json.dump(out, f, indent=2)
print("\nSaved:", path)